# PostgreSQL from a notebook

**First start the database** in a terminal:

```bash
rsm-pg-start
```

Then run these cells: create a table, insert rows, and read them back into a
Polars DataFrame. The connection uses the `PG*` variables the environment sets
(port is per-user on the shared server, so it is read from `$PGPORT`).

In [ ]:
import os
import getpass
from sqlalchemy import create_engine, text
import polars as pl

user = os.environ.get('PGUSER', getpass.getuser())
port = os.environ.get('PGPORT')
db = os.environ.get('PGDATABASE', 'rsm-msba')
url = f'postgresql+psycopg2://{user}@127.0.0.1:{port}/{db}'
engine = create_engine(url)
print('connecting to:', url)

## Create a table and insert rows

In [ ]:
with engine.begin() as con:
    con.execute(text('DROP TABLE IF EXISTS films'))
    con.execute(text('CREATE TABLE films (title text, director text, year int)'))
    con.execute(
        text('INSERT INTO films (title, director, year) VALUES (:t, :d, :y)'),
        [
            {'t': 'Dune: Part Two', 'd': 'Denis Villeneuve', 'y': 2024},
            {'t': 'Oppenheimer', 'd': 'Christopher Nolan', 'y': 2023},
        ],
    )
print('inserted 2 rows')

## Read it back with Polars

In [ ]:
with engine.connect() as con:
    films = pl.read_database('SELECT * FROM films ORDER BY year', connection=con)
films

## Clean up

In [ ]:
with engine.begin() as con:
    con.execute(text('DROP TABLE films'))
print('done')